In [ ]:
!pip install langchain langchain-text-splitters langchain-community bs4

### Habilitando o LangSmith (Opcional)

O Langsmith é uma ferramenta que permite a observabilidade nos nossos sistemas de agentes

- Crie uma conta no langsmith: https://smith.langchain.com/
- Na página principal, após o login, clique em "settings". Depois "API Keys"
- Por fim clique em "+ API Key" e crie uma nova chave do tipo "Personal Access Token"

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

langsmith_key = os.getenv("LANGSMITH_API_KEY")
if not langsmith_key:
    print("API KEY não fornecida")

### Escolhendo o modelo de chat

Estamos usando o Mistral como provedor e o modelo "mistral-small-latest".

É necessário ter uma variável de ambiente (MISTRAL_API_KEY) com uma API Key configurada

In [3]:
'''
Cria objeto para consumir modelos do Mistral. Outros provedores são bem parecidos.
Link da documentação: https://python.langchain.com/api_reference/mistralai/chat_models/langchain_mistralai.chat_models.ChatMistralAI.html#langchain_mistralai.chat_models.ChatMistralAI.get_num_tokens_from_messages
'''

import os
from langchain.chat_models import init_chat_model

model = init_chat_model("mistral-small-latest")

/home/alexandre/Documentos/Unipam/aulas/ia/ia-8p/iaenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model

### Definindo o modelo de embeddings

Aqui escolhemos o modelo que será usado para "vetorizar" a query do usuário e as informações em nossa base de dados

In [5]:
from langchain_mistralai import MistralAIEmbeddings

embeddings = MistralAIEmbeddings(model="mistral-embed")

In [ ]:
embeddings

### Definindo o método de armazenamento de vetores (Vector Store)

Neste exemplo, por motivos de simplicidade, vamos salvar os vetores em memória.

In [ ]:
!pip install -U "langchain-core"

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

### Indexing

Nesta etapa buscamos as informações complementares para o nosso modelo

In [ ]:
'''
Documento loaders. Objetos que nos permitem acessar
e carregas os "documents" onde estão as informações que queremos
'''

import bs4 # Ferramenta para scraping de páginas web.
from langchain_community.document_loaders import WebBaseLoader # Carrega documentos de páginas web.

# Escolhemos apenas algumas classes CSS para evitar carregar menus, rodapés, etc.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 43047


In [13]:
print(docs[0].page_content[:500])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


### Spliting

O texto completo pode ser maior que o contexto do nosso modelo. Por esse motivo é necessário dividir em "chunks" para gerar os vetores

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 63 sub-documents.


In [16]:
all_splits[0]

Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 8}, page_content='LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from 

### Storing

Armazenamos os chunks em memória, já convertidos em vetores. Idealmente **não salvamos em memória mas sim em bancos de dados vetoriais**

In [ ]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

### Retrieval

Criamos uma tool do tipo code-agent que irá executar o processo de recuperação da informação em nossa base vetorial

In [18]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

### Agente

In [ ]:
from langchain.agents import create_agent


tools = [retrieve_context]

prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)
agent = create_agent(model, tools, system_prompt=prompt)

In [ ]:
query = (
    "What is the standard method for Task Decomposition?\n\n"
    "Once you get the answer, look up common extensions of that method."
)

for event in agent.stream( # Streaming response. Permite ver a resposta parcial enquanto é gerada
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the standard method for Task Decomposition?

Once you get the answer, look up common extensions of that method.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (oIM0C7f2Q)
 Call ID: oIM0C7f2Q
  Args:
    query: standard method for task decomposition
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 2578}
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planni